# RAG

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# llm model

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Ingestion Pipeline

In [11]:
# load file

from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("elon_musk_achievements.pdf")

document = pdf_loader.load()

In [12]:
document

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-15T08:22:14+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-15T08:22:14+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'elon_musk_achievements.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Elon Musk and His Achievements\nElon Musk is one of the most influential technology entrepreneurs of the 21st century. He was born on June 28, 1971,\nin Pretoria, South Africa. Musk developed an interest in computers and technology at a very young age and taught\nhimself programming. He later moved to the United States to pursue opportunities in science, engineering, and\nbusiness. One of Musk’s earliest successes came from co-founding Zip2, a software company that provided business\ndirectories and maps for newspapers. The company was eventually sold for nearly $300 million. After that

In [16]:
document[0].metadata

{'producer': 'ReportLab PDF Library - www.reportlab.com',
 'creator': '(unspecified)',
 'creationdate': '2026-05-15T08:22:14+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-05-15T08:22:14+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'elon_musk_achievements.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}

In [24]:
print(f'no of pages: {len(document)}')

no of pages: 3


# Chunks

In [25]:
# chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

docs = text_splitter.split_documents(document)

In [26]:
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-15T08:22:14+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-15T08:22:14+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'elon_musk_achievements.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Elon Musk and His Achievements\nElon Musk is one of the most influential technology entrepreneurs of the 21st century. He was born on June 28, 1971,\nin Pretoria, South Africa. Musk developed an interest in computers and technology at a very young age and taught\nhimself programming. He later moved to the United States to pursue opportunities in science, engineering, and\nbusiness. One of Musk’s earliest successes came from co-founding Zip2, a software company that provided business\ndirectories and maps for newspapers. The company was eventually sold for nearly $300 million. After that

## Embedding

In [28]:
# Embeddings

from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

## Vector DB

In [32]:
# Vector Database (FAISS) 

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embedding)

In [33]:
print(vectorstore.index.ntotal)

6


# Retriver Pipeline 

In [35]:
# Retriever

retriever = vectorstore.as_retriever()

In [36]:
# test

test_query = "who is elon musk?"

retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Elon Musk and His Achievements
Elon Musk is one of the most influential technology entrepreneurs of the 21st century. He was born on June 28, 1971,
in Pretoria, South Africa. Musk developed an interest in computers and technology at a very young age and taught
himself programming. He later moved to 

--- Chunk 2 ---
Influence and Legacy
Beyond SpaceX and Tesla, Elon Musk has contributed to several other innovative projects. He co-founded Neuralink,
a company researching brain-computer interfaces, and The Boring Company, which focuses on tunnel transportation
systems to reduce urban traffic congestion. Musk has 

--- Chunk 3 ---
and space exploration. Many people view Musk as a symbol of technological ambition. His achievements demonstrate
how determination, risk-taking, and long-term vision can transform industries and shape the future. Whether through
electric vehicles, reusable rockets, or futuristic technologies, Elon M

--- Chunk 4 ---
Achievements in Technology and

## langgraph

In [38]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [40]:
# rag node

from langchain_core.messages import HumanMessage,AIMessage

def rag_node(state:State):
    # print(f'State: {state}')
    
    question = state['messages'][-1].content
    # print(f'question: {question}') 

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    history = "\n".join(f"{msg.type}: {msg.content}" for msg in state["messages"][:-1])

    # history = state['messages']
    # print(f'history: {history}')

    prompt = f"""
    Conversation History:
    {history}

    Context:
    {context}

    Question:
    {question}

    Answer ONLY from context and conversation history.
    If not found say:
    I don't know based on the provided documents.
    """

    # Instructions:
    # 1. Use the Context as the primary source of truth.
    # 2. Answer factual questions only from the Context.
    # 3. Use Conversation History only to resolve references (e.g., he, she, it, they, this person, that company, we).
    # 4. Do not use external knowledge.
    # 5. If the answer is not present in the Context, say:
    # I couldn't find that information in the provided context.
    # """

    response = llm.invoke(prompt)
    # print(f'response: {response}')

    return {"messages": [AIMessage(content=response.content)]}
